# Eksperimen Machine Learning - Analisis Sentimen Bahasa Indonesia
**Nama:** Melinda Sevira  
**Dataset:** SmSA (Shopee Multi-domain Sentiment Analysis) - IndoNLU  
**Task:** Klasifikasi Sentimen (Positif, Negatif, Netral)

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import os
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from collections import Counter

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

print('Library berhasil diimport!')

## 2. Data Loading

In [ ]:
# Load dataset dari HuggingFace
dataset = load_dataset('indonlu', 'smsa')
print('Dataset berhasil dimuat!')
print(dataset)

In [ ]:
# Konversi ke DataFrame
df_train = pd.DataFrame(dataset['train'])
df_val   = pd.DataFrame(dataset['validation'])
df_test  = pd.DataFrame(dataset['test'])

# Gabungkan semua split untuk eksplorasi
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f'Total data: {len(df)}')
print(f'Train : {len(df_train)}')
print(f'Val   : {len(df_val)}')
print(f'Test  : {len(df_test)}')

In [ ]:
# Simpan raw dataset
os.makedirs('../smsa_raw', exist_ok=True)
df.to_csv('../smsa_raw/smsa_raw.csv', index=False)
print('Raw dataset disimpan ke ../smsa_raw/smsa_raw.csv')

In [ ]:
# Preview data
df.head(10)

In [ ]:
# Info dasar
print('Shape:', df.shape)
print('\nInfo:')
df.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Cek missing values
print('Missing Values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
# Mapping label
label_map = {0: 'positif', 1: 'netral', 2: 'negatif'}
df['sentiment'] = df['label'].map(label_map)

# Distribusi label
print('Distribusi Label:')
print(df['sentiment'].value_counts())

In [ ]:
# Visualisasi distribusi label
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#95a5a6', '#e74c3c']
df['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Distribusi Label Sentimen', fontsize=14)
axes[0].set_xlabel('Sentimen')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df['sentiment'].value_counts().plot(kind='pie', ax=axes[1], colors=colors,
                                     autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporsi Label Sentimen', fontsize=14)
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('distribusi_label.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot disimpan!')

In [ ]:
# Analisis panjang teks
df['text_length'] = df['text'].apply(len)
df['word_count']  = df['text'].apply(lambda x: len(x.split()))

print('Statistik Panjang Teks (karakter):')
print(df['text_length'].describe())
print('\nStatistik Jumlah Kata:')
print(df['word_count'].describe())

In [ ]:
# Visualisasi distribusi panjang teks per sentimen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment, color in zip(['positif', 'netral', 'negatif'], colors):
    subset = df[df['sentiment'] == sentiment]
    axes[0].hist(subset['text_length'], bins=30, alpha=0.6, label=sentiment, color=color)
    axes[1].hist(subset['word_count'],  bins=30, alpha=0.6, label=sentiment, color=color)

axes[0].set_title('Distribusi Panjang Teks (karakter)')
axes[0].set_xlabel('Panjang Teks')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

axes[1].set_title('Distribusi Jumlah Kata')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.savefig('distribusi_panjang_teks.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Top 20 kata paling sering muncul (sebelum preprocessing)
all_words = ' '.join(df['text'].values).split()
word_freq = Counter(all_words)
top_words = pd.DataFrame(word_freq.most_common(20), columns=['word', 'count'])

plt.figure(figsize=(10, 5))
sns.barplot(data=top_words, x='count', y='word', palette='viridis')
plt.title('Top 20 Kata Paling Sering Muncul (Sebelum Preprocessing)')
plt.xlabel('Frekuensi')
plt.tight_layout()
plt.savefig('top20_kata_raw.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# Daftar stopwords Bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

# Tambahkan stopwords custom
custom_stopwords = {'yg', 'nya', 'ini', 'itu', 'dan', 'di', 'ke', 'dari',
                    'utk', 'tdk', 'gak', 'ga', 'ya', 'aja', 'jg', 'sy',
                    'dgn', 'dg', 'tp', 'tapi', 'sdh', 'udah', 'dah', 'jd'}
stop_words.update(custom_stopwords)

print(f'Total stopwords: {len(stop_words)}')

In [ ]:
def clean_text(text):
    """Membersihkan teks dari noise."""
    # Lowercase
    text = text.lower()
    # Hapus URL
    text = re.sub(r'http\S+|www\S+', '', text)
    # Hapus mention dan hashtag
    text = re.sub(r'@\w+|#\w+', '', text)
    # Hapus angka
    text = re.sub(r'\d+', '', text)
    # Hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_text(text):
    """Tokenisasi teks."""
    return word_tokenize(text)

def remove_stopwords(tokens):
    """Menghapus stopwords dari list token."""
    return [token for token in tokens if token not in stop_words and len(token) > 1]

def preprocess_text(text):
    """Pipeline preprocessing lengkap."""
    text   = clean_text(text)
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens)
    return ' '.join(tokens)

print('Fungsi preprocessing berhasil didefinisikan!')

In [ ]:
# Contoh preprocessing
contoh = df['text'].iloc[0]
print('Sebelum:', contoh)
print('Sesudah:', preprocess_text(contoh))

In [ ]:
# Terapkan preprocessing ke seluruh dataset
print('Melakukan preprocessing...')
df['text_clean'] = df['text'].apply(preprocess_text)
print('Preprocessing selesai!')
df[['text', 'text_clean', 'sentiment']].head(5)

In [ ]:
# Hapus baris dengan teks kosong setelah preprocessing
before = len(df)
df = df[df['text_clean'].str.strip() != '']
df = df.dropna(subset=['text_clean'])
after = len(df)
print(f'Baris dihapus (teks kosong): {before - after}')
print(f'Total data setelah cleaning: {after}')

In [ ]:
# Encode label
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['sentiment'])

print('Label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} -> {cls}')

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X = tfidf.fit_transform(df['text_clean'])
y = df['label_encoded'].values

print(f'Shape TF-IDF matrix: {X.shape}')
print(f'Jumlah fitur: {len(tfidf.get_feature_names_out())}')

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')

In [ ]:
# Visualisasi distribusi kata setelah preprocessing
all_words_clean = ' '.join(df['text_clean'].values).split()
word_freq_clean = Counter(all_words_clean)
top_words_clean = pd.DataFrame(word_freq_clean.most_common(20), columns=['word', 'count'])

plt.figure(figsize=(10, 5))
sns.barplot(data=top_words_clean, x='count', y='word', palette='magma')
plt.title('Top 20 Kata Paling Sering Muncul (Setelah Preprocessing)')
plt.xlabel('Frekuensi')
plt.tight_layout()
plt.savefig('top20_kata_clean.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Simpan hasil preprocessing
os.makedirs('smsa_preprocessing', exist_ok=True)

# Simpan dataframe hasil preprocessing
df_output = df[['text', 'text_clean', 'sentiment', 'label_encoded']].copy()
df_output.to_csv('smsa_preprocessing/smsa_preprocessed.csv', index=False)

print('Dataset preprocessing disimpan ke smsa_preprocessing/smsa_preprocessed.csv')
print('\nPreview:')
df_output.head()

In [ ]:
# Verifikasi hasil
df_check = pd.read_csv('smsa_preprocessing/smsa_preprocessed.csv')
print('Shape:', df_check.shape)
print('Kolom:', list(df_check.columns))
print('\nDistribusi label akhir:')
print(df_check['sentiment'].value_counts())
print('\nPreprocessing selesai dan data siap untuk pelatihan model!')